In [93]:
import pandas as pd
df = pd.read_csv('assets/shot_logs.csv')

### Clean dataset

In [94]:
COLUMNS_TO_DROP = [
    'GAME_ID', 'MATCHUP', 'W', 'FINAL_MARGIN', 'PTS', 'SHOT_RESULT', 'CLOSEST_DEFENDER', 'player_name'
]
df = df.drop(columns=COLUMNS_TO_DROP)

df['LOCATION'] = df['LOCATION'].map({'A': 0, 'H': 1})
df['PTS_TYPE'] = df['PTS_TYPE'].map({2: 0, 3: 1})
df['GAME_CLOCK'] = pd.to_timedelta('00:' + df['GAME_CLOCK']).dt.total_seconds()
df['SHOT_CLOCK'] = df['SHOT_CLOCK'].fillna(24)
df['TOUCH_TIME'] = df['TOUCH_TIME'].clip(lower=0).fillna(0)

In [98]:
Y = df['FGM']
X = df.drop(columns=['FGM'])

### Split train/dev/test datasets

In [123]:
from sklearn.model_selection import train_test_split

# 80%/10%/10% split for train/dev/test datasets
X_trainval, X_test, Y_trainval, Y_test = train_test_split(X, Y, 
    test_size=0.1, 
    random_state=42,
    stratify=Y,
)
X_train, X_dev, Y_train, Y_dev = train_test_split(X_trainval, Y_trainval,
    test_size=0.1111,  # 0.1111 * 0.9 ≈ 0.1, so that dev set is 10% of the total
    random_state=42,
    stratify=Y_trainval,
)

### Target encode high-cardinality (only train dataset)

Deal with `player_id` and `CLOSEST_DEFENDER_PLAYER_ID` replacing each id with that players's mean `FGM`. (Avoid embedding layers for now)

In [126]:
k = 20 # smoothing strength
global_mean = Y_train.mean()

# Calculate the mean FGM for each player_id and closest defender player_id in the training set
encoders = {}
for col in ['player_id', 'CLOSEST_DEFENDER_PLAYER_ID']:
    counts = X_train.groupby(col).size()
    sums = Y_train.groupby(X_train[col]).sum()
    # weighted average of the player's own rate and the global rate
    encoders[col] = (sums + k * global_mean) / (counts + k)

# Map the means to the train, dev, and test sets, filling missing values with the global mean
for col, mean in encoders.items():
    X_train[col] = X_train[col].map(mean)
    X_dev[col] = X_dev[col].map(mean).fillna(global_mean)
    X_test[col] = X_test[col].map(mean).fillna(global_mean)

### Standardize numeric features (only train dataset)

`SHOT_NUMBER`, `PERIOD`, `GAME_CLOCK`, `SHOT_CLOCK`, `DRIBBLES`, `TOUCH_TIME`, `SHOT_DIST`, `CLOSE_DEF_DIST`